1. The Strategy We will create features in three categories:


Domain-Specific: Based on telecom business logic (e.g., tenure groups, service bundles).


Mathematical: Ratios and aggregations (e.g., average monthly spend).


Encoding: Converting categories to numbers safely (One-Hot, Label).

In [50]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/cleaned_churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,CUST00001,Male,0.0,No,Yes,3.0,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,No,Mailed check,68.61,205.83,Yes
1,CUST00002,Male,1.0,Yes,No,2.0,Yes,Yes,DSL,No,...,No internet service,Yes,No internet service,No,One year,Yes,Bank transfer (automatic),23.15,46.30,No
2,CUST00003,Female,0.0,No,No,42.0,Yes,Yes,DSL,No,...,No,No internet service,Yes,Yes,Month-to-month,No,Electronic check,42.63,1790.46,Yes
3,CUST00004,Female,0.0,No,Yes,40.0,Yes,Yes,Fiber optic,No,...,Yes,No,No,No internet service,Month-to-month,No,Electronic check,75.04,3001.60,No
4,CUST00005,Male,1.0,Yes,Yes,17.0,Yes,No,Fiber optic,Yes,...,Yes,No,No internet service,No,Two year,Yes,Electronic check,22.38,380.46,Yes


In [51]:
df_fe = df.copy()

if "customerID" in df_fe.columns:
    df_fe = df_fe.drop(columns=["customerID"])

print("Remaining columns after dropping ID columns:")
print(df_fe.columns.tolist())

Remaining columns after dropping ID columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [52]:
# --- Safety Checks ---
# Ensure numeric columns are actually numeric
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
for col in numeric_cols:
    if col in df_fe.columns:
        df_fe[col] = pd.to_numeric(df_fe[col], errors='coerce')

# Ensure Churn is binary (0/1)
if 'Churn' in df_fe.columns and df_fe['Churn'].dtype == 'object':
    df_fe['Churn'] = df_fe['Churn'].map({'Yes': 1, 'No': 0})

print(f"Data loaded: {df_fe.shape}")

Data loaded: (61432, 20)


In [53]:
# 1. Value Risk Ratio
# Business Logic: High monthly charges with low tenure indicates high churn risk.
df_fe['value_risk_ratio'] = df_fe['MonthlyCharges'] / (df_fe['tenure'] + 1)

# 2. Average Monthly Spend
# Business Logic: Normalizes total spend by lifespan.
df_fe['avg_monthly_spend'] = df_fe['TotalCharges'] / (df_fe['tenure'] + 1)

# 3. Tenure Groups (Binning)
# Business Logic: Churn risk is non-linear (first 6 months are critical).
bins = [0, 6, 12, 24, 48, float('inf')]
labels = ['0-6 Months', '6-12 Months', '1-2 Years', '2-4 Years', '4+ Years']
df_fe['tenure_group'] = pd.cut(df_fe['tenure'], bins=bins, labels=labels, include_lowest=True)

# 4. New Customer Flag
df_fe['is_new_customer'] = (df_fe['tenure'] <= 6).astype(int)

print("Numerical features created.")

Numerical features created.


In [54]:
#Create service count feature
service_cols = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]


# Ensure columns exist before applying
available_services = [col for col in service_cols if col in df_fe.columns]

        
df_fe["service_count"] = df_fe[available_services].apply(
    lambda row: sum(str(v).lower() == "yes" for v in row),
    axis=1
)

# Business Logic: Customers with security/tech support are less likely to leave due to issues.
if "OnlineSecurity" in df_fe.columns and "TechSupport" in df_fe.columns:
    df_fe["has_support_services"] = (
        (df_fe["OnlineSecurity"] == "Yes") |
        (df_fe["TechSupport"] == "Yes")
    ).astype(int)
else:
    df_fe["has_support_services"] = 0

print("Service features created.")

# df_fe['MonthlyCharges_scaled'] = (df_fe['MonthlyCharges'] - df_fe['MonthlyCharges'].mean()) / df_fe['MonthlyCharges'].std()
# df_fe['service_count_scaled'] = df_fe['service_count'] / df_fe['service_count'].max()
# # df_fe['engagement_vs_cost'] = df_fe['MonthlyCharges_scaled'] * df_fe['service_count_scaled']
# # df_fe['engagement_vs_cost_variance'] = df_fe['charge_variance'] * df_fe['service_count_scaled']

# df_fe['service_bin'] = pd.cut(
#     df_fe['service_count'],
#     bins=[-1, 2, 4, 6],
#     labels=['Low','Medium','High']
# )

# df_fe['engagement_vs_cost_bin'] = pd.qcut(
#     df_fe['MonthlyCharges'] * (df_fe['service_count'] / df_fe['service_count'].max()), 
#     4, duplicates='drop', 
#     labels=['Low','Medium','High']
# )

# df_fe['engagement_risk_score'] = (df_fe['MonthlyCharges_scaled'] * df_fe['service_count_scaled']) * df_fe['tenure']

# df_fe['engagement_risk_score'].head()
# # display(df_fe["service_count"].describe())

Service features created.


In [55]:
###### I removed these features for now cuz we gonna encode them later #####

# Paperless Billing Flag
if "PaperlessBilling" in df_fe.columns:
    df_fe["is_paperless"] = (df_fe["PaperlessBilling"] == "Yes").astype(int)
else:
    df_fe["is_paperless"] = 0

print("Contract & Payment features created.")


Contract & Payment features created.


In [56]:
df_fe.to_csv('../data/processed/telco_features_CatBoost.csv', index=False)

In [57]:
# Map Yes/No to 1/0 safely
binary_map = {"Yes": 1, "No": 0}

binary_cols = [
    "Partner", "Dependents", "PhoneService", "PaperlessBilling",
    "OnlineSecurity", "OnlineBackup", "DeviceProtection", 
    "TechSupport", "StreamingTV", "StreamingMovies"
]

for col in binary_cols:
    if col in df_fe.columns:
        # Map values, fill NaN with -1 to flag potential data issues
        df_fe[col] = df_fe[col].map(binary_map).fillna(-1)

# Gender Encoding
if "gender" in df_fe.columns:
    df_fe["gender"] = df_fe["gender"].map({"Male": 1, "Female": 0}).fillna(-1)

print("Binary encoding complete.")

Binary encoding complete.


In [58]:
# Identify columns to encode
# This reduces multicollinearity and dimensionality.

categorical_to_encode = [
    "MultipleLines", "InternetService", 
    "tenure_group", "Contract", "PaymentMethod"  # Encode the binned tenure groups
]

# Filter to only columns that exist
categorical_to_encode = [col for col in categorical_to_encode if col in df_fe.columns]

# Apply One-Hot Encoding
df_model = pd.get_dummies(df_fe, columns=categorical_to_encode, drop_first=True)

print(f"✅ One-hot encoding complete. New shape: {df_model.shape}")


✅ One-hot encoding complete. New shape: (61432, 36)


In [59]:
bool_cols = df_model.select_dtypes(include='bool').columns

df_model[bool_cols] = df_model[bool_cols].astype(int)

In [60]:
#Check final data types
print("Final data types:")
display(df_model.dtypes)

Final data types:


gender                                     int64
SeniorCitizen                            float64
Partner                                    int64
Dependents                                 int64
tenure                                   float64
PhoneService                               int64
OnlineSecurity                           float64
OnlineBackup                             float64
DeviceProtection                         float64
TechSupport                              float64
StreamingTV                              float64
StreamingMovies                          float64
PaperlessBilling                           int64
MonthlyCharges                           float64
TotalCharges                             float64
Churn                                      int64
value_risk_ratio                         float64
avg_monthly_spend                        float64
is_new_customer                            int64
service_count                              int64
has_support_services

In [25]:
#Create average charge per month
df_model["avg_charge_per_month"] = df_model["TotalCharges"] / (df_model["tenure"] + 1)

#Charge Variance
# Business Logic: Indicates if recent monthly charges are higher than their historical average.
df_model['charge_variance'] = df_model['MonthlyCharges'] - df_model['avg_charge_per_month']

# print("Summary of avg_charge_per_month:")
# display(df_model["avg_charge_per_month"].describe())
df_model.shape
# print(df_model[["MonthlyCharges", "avg_charge_per_month", "charge_variance"]].head())

(61432, 38)

In [62]:
#Check missing values after feature engineering
missing_after_fe = df_model.isna().sum().sort_values(ascending=False)
missing_after_fe = missing_after_fe[missing_after_fe > 0]

print("Remaining missing values:")
display(missing_after_fe)

Remaining missing values:


Series([], dtype: int64)

In [32]:
# 1. Check for remaining object columns
remaining_objects = df_model.select_dtypes(include='object').columns.tolist()
if remaining_objects:
    print(f"⚠️ Warning: Remaining object columns: {remaining_objects}")
else:
    print("✅ All features are numeric.")

# 2. Check for NaN values
nan_count = df_model.isnull().sum().sum()
if nan_count > 0:
    print(f"⚠️ Warning: {nan_count} missing values found.")
else:
    print("✅ No missing values.")


# 4. Save Final Dataset
if 'customerID' in df_model.columns:
    df_final = df_model.drop('customerID', axis=1)
else:
    df_final = df_model

df_final.to_csv('../data/processed/telco_features.csv', index=False)
print("📄 Saved: ../data/processed/telco_features.csv")
print(f"Final dataset shape: {df_final.shape}")

✅ All features are numeric.
✅ No missing values.
📄 Saved: ../data/processed/telco_features.csv
Final dataset shape: (61432, 38)
